# **Import**

In [1]:
import pandas as pd
import numpy as np

diabetes = pd.read_csv('diabetic_data.csv')

# **Baselines**

In [2]:
diabetes['admission_type_id'] = diabetes['admission_type_id'].replace(6, np.nan)
diabetes['discharge_disposition_id'] = diabetes['discharge_disposition_id'].replace(18, np.nan)
diabetes['admission_source_id'] = diabetes['admission_source_id'].replace(17, np.nan)

diabetes.replace('?', np.nan, inplace=True)

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def reclassify_readmitted(status):
    if status == 'NO':
        return 'Not Readmitted'
    else:
        return 'Readmitted'

diabetes['readmitted'] = diabetes['readmitted'].apply(reclassify_readmitted)

# Label encoding
le_readmitted = LabelEncoder()
diabetes['readmitted_encoded'] = le_readmitted.fit_transform(diabetes['readmitted'])

# Drop the original 'readmitted' column as it's been encoded
diabetes = diabetes.drop(columns=['readmitted'])

# Define numerical and categorical columns
numerical_cols = diabetes.select_dtypes(include=np.number).columns.tolist()
object_cols = diabetes.select_dtypes(include='object').columns.tolist()

# Exclude target column from numerical_cols
if 'readmitted_encoded' in numerical_cols:
    numerical_cols.remove('readmitted_encoded')

# 4. Impute any remaining NaN values in numerical columns with their respective column means
for col in numerical_cols:
    if diabetes[col].isnull().any():
        diabetes[col] = diabetes[col].fillna(diabetes[col].mean())

for col in object_cols:
    diabetes[col] = diabetes[col].fillna('Unknown')
    le = LabelEncoder()
    diabetes[col] = le.fit_transform(diabetes[col])
    print(f" - Encoded '{col}'")


diabetes.head()
print("\nRemaining Null values after processing:")
print(diabetes.isnull().sum()[diabetes.isnull().sum() > 0])

 - Encoded 'race'
 - Encoded 'gender'
 - Encoded 'age'
 - Encoded 'weight'
 - Encoded 'payer_code'
 - Encoded 'medical_specialty'
 - Encoded 'diag_1'
 - Encoded 'diag_2'
 - Encoded 'diag_3'
 - Encoded 'max_glu_serum'
 - Encoded 'A1Cresult'
 - Encoded 'metformin'
 - Encoded 'repaglinide'
 - Encoded 'nateglinide'
 - Encoded 'chlorpropamide'
 - Encoded 'glimepiride'
 - Encoded 'acetohexamide'
 - Encoded 'glipizide'
 - Encoded 'glyburide'
 - Encoded 'tolbutamide'
 - Encoded 'pioglitazone'
 - Encoded 'rosiglitazone'
 - Encoded 'acarbose'
 - Encoded 'miglitol'
 - Encoded 'troglitazone'
 - Encoded 'tolazamide'
 - Encoded 'examide'
 - Encoded 'citoglipton'
 - Encoded 'insulin'
 - Encoded 'glyburide-metformin'
 - Encoded 'glipizide-metformin'
 - Encoded 'glimepiride-pioglitazone'
 - Encoded 'metformin-rosiglitazone'
 - Encoded 'metformin-pioglitazone'
 - Encoded 'change'
 - Encoded 'diabetesMed'

Remaining Null values after processing:
Series([], dtype: int64)


In [4]:
from sklearn.model_selection import train_test_split

X = diabetes.drop('readmitted_encoded', axis=1)
y = diabetes['readmitted_encoded']

# Training Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# LogisticRegression model
log_reg_minimal = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)

# Fit
log_reg_minimal.fit(X_train, y_train)
y_pred_lr_minimal = log_reg_minimal.predict(X_test)

# Classification report
print("\nClassification Report (Logistic Regression on minimally processed data):")
print(classification_report(y_test, y_pred_lr_minimal))


Classification Report (Logistic Regression on minimally processed data):
              precision    recall  f1-score   support

           0       0.57      0.76      0.65     10973
           1       0.54      0.33      0.41      9381

    accuracy                           0.56     20354
   macro avg       0.56      0.55      0.53     20354
weighted avg       0.56      0.56      0.54     20354



In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Instantiate a DecisionTreeClassifier
dtc_minimal = DecisionTreeClassifier(random_state=42)

# Fit
dtc_minimal.fit(X_train, y_train)

# Make predictions
y_pred_dtc_minimal = dtc_minimal.predict(X_test)

# Classification report
print("\nClassification Report (Decision Tree Classifier on minimally processed data):")
print(classification_report(y_test, y_pred_dtc_minimal))


Classification Report (Decision Tree Classifier on minimally processed data):
              precision    recall  f1-score   support

           0       0.61      0.61      0.61     10973
           1       0.55      0.55      0.55      9381

    accuracy                           0.58     20354
   macro avg       0.58      0.58      0.58     20354
weighted avg       0.58      0.58      0.58     20354



In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Instantiate a RandomForestClassifier
rfc_minimal = RandomForestClassifier(random_state=42, n_estimators=100)

# Fit
rfc_minimal.fit(X_train, y_train)
y_pred_rfc_minimal = rfc_minimal.predict(X_test)

# Classification report
print("\nClassification Report (Random Forest Classifier on minimally processed data):")
print(classification_report(y_test, y_pred_rfc_minimal))


Classification Report (Random Forest Classifier on minimally processed data):
              precision    recall  f1-score   support

           0       0.67      0.74      0.71     10973
           1       0.66      0.58      0.62      9381

    accuracy                           0.67     20354
   macro avg       0.67      0.66      0.66     20354
weighted avg       0.67      0.67      0.66     20354



In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

gbc_minimal = GradientBoostingClassifier(random_state=42, n_estimators=100)

gbc_minimal.fit(X_train, y_train)
y_pred_gbc_minimal = gbc_minimal.predict(X_test)

# Classification report
print("\nClassification Report (Gradient Boosting Classifier on minimally processed data):")
print(classification_report(y_test, y_pred_gbc_minimal))


Classification Report (Gradient Boosting Classifier on minimally processed data):
              precision    recall  f1-score   support

           0       0.67      0.73      0.70     10973
           1       0.65      0.59      0.62      9381

    accuracy                           0.67     20354
   macro avg       0.66      0.66      0.66     20354
weighted avg       0.66      0.67      0.66     20354



In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# KNeighborsClassifier
knn_minimal = KNeighborsClassifier(n_neighbors=5)

# Fit
knn_minimal.fit(X_train, y_train)
y_pred_knn_minimal = knn_minimal.predict(X_test)

# Classification report
print("\nClassification Report (K-Nearest Neighbors on minimally processed data):")
print(classification_report(y_test, y_pred_knn_minimal))


Classification Report (K-Nearest Neighbors on minimally processed data):
              precision    recall  f1-score   support

           0       0.59      0.62      0.60     10973
           1       0.53      0.50      0.52      9381

    accuracy                           0.56     20354
   macro avg       0.56      0.56      0.56     20354
weighted avg       0.56      0.56      0.56     20354



# **Cleaning**

### Replace necessary IDs with Null

In [10]:
diabetes['admission_type_id'] = diabetes['admission_type_id'].replace(6, np.nan)
diabetes['discharge_disposition_id'] = diabetes['discharge_disposition_id'].replace(18, np.nan)
diabetes['admission_source_id'] = diabetes['admission_source_id'].replace(17, np.nan)
diabetes.replace('?', np.nan, inplace=True)

### Drop Unknown Gender, Race. Mid Point Age

In [11]:
# Drop rows with 'Unknown/Invalid' gender
if 'Unknown/Invalid' in diabetes['gender'].unique():
    diabetes = diabetes[diabetes['gender'] != 'Unknown/Invalid']

# Drop rows with unknown race '?'
if '?' in diabetes['race'].unique():
    diabetes = diabetes[diabetes['race'] != '?']
diabetes.dropna(subset=['race'], inplace=True)

# Convert age ranges to midpoints
age_ranges = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
    '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
diabetes['age'] = diabetes['age'].replace(age_ranges)

### Remaining Features

In [12]:
diabetes.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted_encoded']

# **LightGBM on Processed Dataset (RUS)**

In [17]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler

df_lgbm = diabetes.copy() # copy

X_lgbm = df_lgbm.drop('readmitted', axis=1)
y_lgbm = df_lgbm['readmitted']

X_train_lgbm, X_test_lgbm, y_train_lgbm, y_test_lgbm = train_test_split(X_lgbm, y_lgbm, test_size=0.2, random_state=42, stratify=y_lgbm)
rus = RandomUnderSampler(random_state=42)
X_resampled_lgbm, y_resampled_lgbm = rus.fit_resample(X_train_lgbm, y_train_lgbm)

# ----------------------------------------------------------------------------------------------------------------------------------------

# LGBMClassifier
lgbm_undersampled_classifier = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    random_state=42,
    n_estimators=100,
    categorical_feature=categorical_features_indices_lgbm
)

print("Training LightGBM Classifier with undersampled data...")
lgbm_undersampled_classifier.fit(X_resampled_lgbm, y_resampled_lgbm)

y_pred_lgbm_undersampled = lgbm_undersampled_classifier.predict(X_test_lgbm)

print("\nClassification Report (LightGBM - Undersampled):")
print(classification_report(y_test_lgbm, y_pred_lgbm_undersampled, target_names=target_names_for_report))

Training LightGBM Classifier with undersampled data...


/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(


[LightGBM] [Info] Number of positive: 37521, number of negative: 37521
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.147708 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1711
[LightGBM] [Info] Number of data points in the train set: 75042, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

Classification Report (LightGBM - Undersampled):
                precision    recall  f1-score   support

Not Readmitted       0.71      0.66      0.68     10973
    Readmitted       0.63      0.68      0.65      9381

      accuracy                           0.67     20354
     macro avg       0.67      0.67      0.67     20354
  weighted avg       0.67      0.67      0.67     20354

